In [30]:
import numpy as np
from scipy.integrate import solve_ivp
import pykoopman as pk

In [31]:
def lorenz(t, s, sigma=10, rho=28, beta=8/3):
    x, y, z = s
    return [sigma*(y - x),
            x*(rho - z) - y,
            x*y - beta*z]

In [32]:
sol = solve_ivp(
    lorenz,
    [0, 60],
    [1, 0, 0],
    t_eval=np.linspace(0, 60, 1000),
    method='RK45'
)

In [33]:
x = sol.y.T

In [34]:
split = int(0.8 * len(x))

In [35]:
x_train = x[:split]
x_test = x[split:]

In [36]:
model = pk.Koopman(
    observables=pk.observables.Polynomial(degree=2, include_bias=True),
)

In [37]:
model.fit(x_train)

/Users/sadi_/Coding/koopman/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:103: FutureWarning: The attribute `n_input_features_` was deprecated in version 1.0 and will be removed in 1.2.
  warnings.warn(msg, category=FutureWarning)
/Users/sadi_/Coding/koopman/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:103: FutureWarning: The attribute `n_input_features_` was deprecated in version 1.0 and will be removed in 1.2.
  warnings.warn(msg, category=FutureWarning)
/Users/sadi_/Coding/koopman/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:103: FutureWarning: The attribute `n_input_features_` was deprecated in version 1.0 and will be removed in 1.2.
  warnings.warn(msg, category=FutureWarning)
/Users/sadi_/Coding/koopman/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:103: FutureWarning: The attribute `n_input_features_` was deprecated in version 1.0 and will be removed in 1.2.
  warnings.warn(msg, category=FutureWarning)


Koopman(observables=Polynomial(),
        regressor=PyDMDRegressor(regressor=<pydmd.dmd.DMD object at 0x169c378d0>))

In [38]:
x_curr = x_test[0: 1]
pred = [x_test[0]]

In [ ]:
for _ in range(50):
    x_current = model.predict(x_curr)
    pred.append(x_current[0])


In [40]:
true  = x_test[:51]
pred = np.array(pred)

In [42]:
rmse = np.sqrt(np.mean((pred - true)**2, axis=1))

In [43]:
print(f"\n{'Step':>5} | {'RMSE':>8}")
print("-" * 18)
for i in range(0, 51, 5):
    print(f"{i:>5} | {rmse[i]:>8.4f}")


 Step |     RMSE
------------------
    0 |   0.0000
    5 |   5.5650
   10 |  18.8950
   15 |   9.0360
   20 |  18.5202
   25 |   8.4561
   30 |  18.3907
   35 |   9.9468
   40 |  11.3953
   45 |  14.4930
   50 |   5.1792
